# Flash attention
## 搭建环境
在ubuntu虚拟机上搭建了环境，可以正确编译以及cuda程序，实现cuda程序的自动跳转，方便编写cuda代码
因此编写代码就放在虚拟机上

因为flash attention2 3 需要利用特殊的硬件特性，因此需要在租相应的平台，在平台上搭建环境很麻烦

先看看autodl，能不能采用docker的方式运行
## 实现flash attention
### 实现层次
需要先看看怎么实现，应该是直接实现flash attention 算子，包括对应的forward、backward

上层的transfomer 模块相应的调用算子（看一下以前的transformer实现）

最后确认了实现的层次：Ops -> CUDA Kernel -> Pybind -> Op Wrapper -> Module Call
### 实现方法
应该用什么实现呢？cuda、triton、cuTile

使用cuda编程的话，应该借用cutblass模版库进行编程

如果使用triton实现的话，绕过了pybind，但是将cuda后端管理的ptr给triton模块，然后再在triton层面进行编程

使用cuTile编程的话，依旧是python编程，和triton类似

### 测试程序
需要先编写测试程序，首先是flash attention 算子的测试程序，该测试程序需要包括一下几个方面

1.直接调用cuda程序中的实现进行测试（确定在ops层面进行测试）

2.需要验证算子的正确性，那和什么进行比对呢（或者在python层面进行比对正确性，参考一下以前的比较）

3.每个测试应该进行多次迭代，从而可以测试时间，包括不同seq长度

（可以参考官方的flash attention实现，看看它是怎么进行测试的）





接下来要做的事：

在autodl平台上跑一次

了解cuTile，决定是用cutlass还是用cuTile

看看别人的实现以及怎么测试、怎么benchmark

backward 不应该都是tensor操作吗，直接调用Narray api不就构建不了计算图了吗？（需要构建一个ops用来计算backward）

测试的时候是调用ops测试，还是module呢，以及怎么获得时间


In [1]:
import sys
print(sys.executable)

/home/xyx/needle_env/bin/python


In [2]:
!pip3 install pybind11

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.6/293.6 kB 539.2 kB/s eta 0:00:00a 0:00:01


In [2]:
!make

-- Found pybind11: /home/xyx/needle_env/lib/python3.12/site-packages/pybind11/include (found version "3.0.1")
-- Found cuda, building cuda backend
-- Configuring done (0.2s)
-- Generating done (0.0s)
-- Build files have been written to: /home/xyx/needle/build
make[1]: Entering directory '/home/xyx/needle/build'
make[2]: Entering directory '/home/xyx/needle/build'
make[3]: Entering directory '/home/xyx/needle/build'
make[3]: Leaving directory '/home/xyx/needle/build'
[ 50%] Built target ndarray_backend_cpu
make[3]: Entering directory '/home/xyx/needle/build'
make[3]: Leaving directory '/home/xyx/needle/build'
[100%] Built target ndarray_backend_cuda
make[2]: Leaving directory '/home/xyx/needle/build'
make[1]: Leaving directory '/home/xyx/needle/build'


In [3]:
%set_env PYTHONPATH ./python
%set_env NEEDLE_BACKEND nd

env: PYTHONPATH=./python
env: NEEDLE_BACKEND=nd


In [4]:
import sys
sys.path.append('./python')

## 测试
### 正确性验证
如果在NDArray层面去测试的话，需要借用numpy来进行比较，而numpy没有现有的api，根据已知的qkv计算self attention，因此需要创建numpy的qkv，然后根据self attention的定义去计算，再与cuda 的NDArray计算结果进行比较，比较麻烦。

因此本测试在flash attention module层面进行正确性验证，仿照已有测试中的attention_activation 测试编写，将flash attention的结果与已知的label进行对比,以及将结果和torch对比。与torch对比的测试用例 sequence length比较长，符合实际。
### benchmark
对于benchmark说，需要计算TFLOPS。先对GPU进行预热，再调用算子层面的flashattention进行计算，迭代50次，计算时间以及总的操作数，从而计算TFOPS。当前的计算基于causal = false，dropout = 0

In [ ]:
!python3 -m pytest tests/hw4/test_transformer.py -l -v -k "attention_activation_vs_torch"

In [ ]:
# 正确性验证
!python3 -m pytest tests/project/test_flashattention.py -l -v -k "test_flashattention_activation and False and 0.0"
!python3 -m pytest tests/project/test_flashattention.py -l -v -k "test_attention_activation_vs_torch and False and 0.0"

In [ ]:
# 计算TFLOPS
!python3 tests/project/benchmark.py --batch_size 8 --num_heads 12 --seq_len 1024 --head_dim 64 --dropout 0.0

In [ ]:
# baseline
!python3 tests/project/benchmark_torch.py --batch_size 8 --num_heads 12 --seq_len 1024 --head_dim 64 --dropout 0.0

In [ ]:
# ncu profile
!ncu --launch-skip 10 --launch-count 1